## Model

### Tree

In [1]:
import numpy as np

class DecisionTreeNode:
    def __init__(self, feature=None, threshold=None, left=None, right=None, label=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.label = label


class CustomDecisionTree:
    def __init__(self, max_depth=10, min_samples=5, n_features=None , min_samples_leaf=1):
        self.max_depth = max_depth
        self.min_samples = min_samples
        self.n_features = n_features
        self.min_samples_leaf = min_samples_leaf
        self.root = None

    def fit(self, X, y):
        self.n_classes = len(np.unique(y))
        self.n_features_total = X.shape[1]
        self.n_features = self.n_features or self.n_features_total
        self.root = self._build_tree(X, y, depth=0)

    def _build_tree(self, X, y, depth):
        n_samples = X.shape[0]

        # stopping conditions
        if (
            depth >= self.max_depth or
            n_samples < self.min_samples or
            len(np.unique(y)) == 1
        ):
            return DecisionTreeNode(label=self._majority_label(y))

        feature_idxs = np.random.choice(self.n_features_total, self.n_features, replace=False)

        best_feature, best_threshold = self._best_split(X, y, feature_idxs)

        if best_feature is None:
            return DecisionTreeNode(label=self._majority_label(y))

        left_mask = X[:, best_feature] < best_threshold
        right_mask = ~left_mask

        left = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return DecisionTreeNode(best_feature, best_threshold, left, right)

    def _best_split(self, X, y, feature_idxs):
        best_gini = float("inf")
        best_feature, best_threshold = None, None

        for feature in feature_idxs:
            X_col = X[:, feature]
            unique_vals = np.unique(X_col)

            if len(unique_vals) == 1:
                continue

            thresholds = (unique_vals[:-1] + unique_vals[1:]) / 2

            step = max(1, len(thresholds) // 10)

            for t in thresholds[::step]:
                gini = self._gini_split(y, X_col, t)

                if gini < best_gini:
                    best_gini = gini
                    best_feature = feature
                    best_threshold = t

        return best_feature, best_threshold

    def _gini_split(self, y, X_col, threshold):
        left = y[X_col < threshold]
        right = y[X_col >= threshold]

        if len(left) < self.min_samples_leaf or len(right) < self.min_samples_leaf:
            return float("inf")

        def gini(group):
            if len(group) == 0:
                return 0

            classes, counts = np.unique(group, return_counts=True)

            probs = counts / np.sum(counts)
            return 1 - np.sum(probs ** 2)

        n = len(y)
        return (len(left)/n)*gini(left) + (len(right)/n)*gini(right)

    def _majority_label(self, y):
        classes, counts = np.unique(y, return_counts=True)
        return classes[np.argmax(counts)]

    def predict(self, X):
        return np.array([self._traverse(x, self.root) for x in X])

    def _traverse(self, x, node):
        if node.label is not None:
            return node.label

        if x[node.feature] < node.threshold:
            return self._traverse(x, node.left)
        else:
            return self._traverse(x, node.right)

### Random Forest

A Random Forest trains multiple decision trees, each on a different bootstrap sample (random sample with replacement) of the training data, with each tree only considering a random subset of features at every split. At prediction time, all trees vote and the majority class wins — this averaging of many decorrelated trees reduces the variance that a single tree suffers from.

In [2]:

class RandomForest:
    def __init__(self, n_trees=10, max_depth=10, min_samples=5, n_features=None, min_samples_leaf=1):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples = min_samples
        self.n_features = n_features
        self.min_samples_leaf = min_samples_leaf
        self.trees = []

    def fit(self, X, y):
        self.trees = []
        n_samples = X.shape[0]

        for _ in range(self.n_trees):
            indices = np.random.choice(n_samples, n_samples, replace=True)
            X_sample = X[indices]
            y_sample = y[indices]

            tree = CustomDecisionTree(
                max_depth=self.max_depth,
                min_samples=self.min_samples,
                n_features=self.n_features,
                min_samples_leaf=self.min_samples_leaf
            )

            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        tree_preds = np.array([tree.predict(X) for tree in self.trees])

        final_preds = []
        for i in range(X.shape[0]):
            votes = tree_preds[:, i]
            final_preds.append(self._majority_vote(votes))

        return np.array(final_preds)

    def _majority_vote(self, votes):
        classes, counts = np.unique(votes, return_counts=True)
        return classes[np.argmax(counts)]

In [ ]:
import importlib
import preprocessing2
importlib.reload(preprocessing2)

## Cross Validation

In [ ]:
from preprocessing2 import preprocess, k_fold_indices, custom_macro_f1_score
import numpy as np

feature_methods = ["flatten", "hog", "pca", "cnn", "pca+hog"]

best_f1_score = 0
best_params = {}

current_run = 1

for feature_method in feature_methods:


    X_train, y_train, X_val, y_val, X_test, y_test, _ = preprocess(
        feature_method=feature_method,
        n_pca=50
    )

    folds = k_fold_indices(X_train, k=3)

    param_grid = {
        'n_trees': [5, 10, 20,50],
        'n_features': [int(np.sqrt(X_train.shape[1])), X_train.shape[1] // 2]
    }

    total_runs = (
        len(feature_methods) *
        len(param_grid['n_trees']) *
        len(param_grid['n_features'])
    )

    for n_trees in param_grid['n_trees']:
                for n_feat in param_grid['n_features']:

                    print(f"--- Run {current_run}/{total_runs} | Feature:{feature_method} | Trees:{n_trees} | n_feat:{n_feat} ---")

                    fold_f1_scores = []

                    for train_idx, val_idx in folds:
                        X_fold_train, y_fold_train = X_train[train_idx], y_train[train_idx]
                        X_fold_val, y_fold_val = X_train[val_idx], y_train[val_idx]

                        cv_model = RandomForest(
                            n_trees=n_trees,
                            max_depth=12,
                            min_samples=10,
                            min_samples_leaf=3,
                            n_features=min(n_feat, X_train.shape[1]),
                        )

                        cv_model.fit(X_fold_train, y_fold_train)
                        preds = cv_model.predict(X_fold_val)

                        fold_f1 = custom_macro_f1_score(y_fold_val, preds, n_classes=10)
                        fold_f1_scores.append(fold_f1)

                    avg_f1 = np.mean(fold_f1_scores)
                    print(f"    -> 3-Fold Average Macro F1: {avg_f1:.4f}\n")

                    if avg_f1 > best_f1_score:
                        best_f1_score = avg_f1
                        best_params = {
                            'feature_method': feature_method,
                            'n_trees': n_trees,
                            'n_features': n_feat
                        }

                    current_run += 1

print("="*50)
print("  RANDOM FOREST GRID SEARCH COMPLETE")
print("="*50)
print(f"Best CV Macro F1: {best_f1_score:.4f}")
print(f"Best Parameters: {best_params}")

## Final Evaluation with best parameters

### Hog Extraction

In [3]:
from skimage.feature import hog
import numpy as np
import time

from preprocessing2 import preprocess

X_train, y_train, X_val, y_val, X_test, y_test, weights = preprocess(
    feature_method="flatten",
    n_pca=None
)

def extract_hog_features(X):

    hog_features = []

    for img in X:

        img_2d = img.reshape(28, 28)

        fd = hog(
            img_2d,
            orientations=9,
            pixels_per_cell=(4, 4),
            cells_per_block=(2, 2),
            block_norm='L2-Hys'
        )

        hog_features.append(fd)

    return np.array(hog_features)

print("Extracting HOG features...")

start = time.time()

X_train = extract_hog_features(X_train)
X_val   = extract_hog_features(X_val)
X_test  = extract_hog_features(X_test)

print(f"HOG extraction completed in {(time.time() - start):.2f} seconds")

print("HOG feature shape:", X_train.shape)

Split completed: Train=54000, Val=6000, Test=10000
Extracting HOG features...
HOG extraction completed in 29.02 seconds
HOG feature shape: (54000, 1296)


### Training

In [5]:

# 2. Initialize model
print("Initializing Custom Decision Tree...")
tree = RandomForest(
    n_trees=20,
    max_depth=12,
    min_samples=2,
    min_samples_leaf=3,
    n_features=int(np.sqrt(X_train.shape[1])),
)

# 3. Train model
print("Training the model...")
start_time = time.time()
tree.fit(X_train, y_train)
print(f"Training completed in {(time.time() - start_time):.2f} seconds.")

Initializing Custom Decision Tree...
Training the model...
Training completed in 256.52 seconds.


## Testing

In [6]:
from preprocessing2 import custom_classification_report, custom_confusion_matrix, custom_accuracy_score


print("\n" + "="*45)
print("  CUSTOM MODEL VALIDATION PERFORMANCE")
print("="*45)

val_preds = tree.predict(X_val)

target_names = [str(i) for i in range(10)]
print(custom_classification_report(y_val, val_preds, target_names=target_names))

val_acc = custom_accuracy_score(y_val, val_preds)
print(f"Validation Accuracy: {val_acc:.4f}")

cm = custom_confusion_matrix(y_val, val_preds)

print("\nValidation Confusion Matrix (rows = actual, cols = predicted):\n")

labels = [str(i) for i in range(10)]

print(f"{'':12}", end="")
for label in labels:
    print(f"{label:>6}", end="")
print()

for i, row in enumerate(cm):
    print(f"{labels[i]:>10} ", end="")
    for val in row:
        print(f"{val:6}", end="")
    print()

print("\n" + "="*45)
print("  FINAL TEST PERFORMANCE")
print("="*45)

test_preds = tree.predict(X_test)

print(custom_classification_report(y_test, test_preds, target_names=target_names))

test_acc = custom_accuracy_score(y_test, test_preds)
print(f"Test Accuracy: {test_acc:.4f}")

cm_test = custom_confusion_matrix(y_test, test_preds)

print("\nTest Confusion Matrix (rows = actual, cols = predicted):\n")

print(f"{'':12}", end="")
for label in labels:
    print(f"{label:>6}", end="")
print()

for i, row in enumerate(cm_test):
    print(f"{labels[i]:>10} ", end="")
    for val in row:
        print(f"{val:6}", end="")
    print()


  CUSTOM MODEL VALIDATION PERFORMANCE
                 precision     recall   f1-score    support

0                     0.98       0.98       0.98        587
1                     0.98       0.99       0.98        630
2                     0.95       0.97       0.96        600
3                     0.93       0.96       0.95        627
4                     0.96       0.96       0.96        595
5                     0.97       0.93       0.95        549
6                     0.98       0.98       0.98        571
7                     0.98       0.96       0.97        668
8                     0.96       0.95       0.95        597
9                     0.95       0.94       0.94        576

accuracy                                    0.96       6000
macro avg             0.96       0.96       0.96       6000

Validation Accuracy: 0.9625

Validation Confusion Matrix (rows = actual, cols = predicted):

                 0     1     2     3     4     5     6     7     8     9
         0  